# Model Pipeline Walkthrough & Zero-Shot VLM Re-Ranking Explorer

This notebook does two things:
1. **Part 1 — Full "Path of the Image" (without VLM):** visualizes every transformation inside our best model (`Original -> DINOv2 PCA Features -> Stage-1 Coarse Match & Crop Boxes -> Stage-2 Zoomed Crops & 92x92 Similarity Map -> Final Prediction`).
2. **Part 2 — What a Zero-Shot Pretrained VLM Would Do (No Fine-Tuning Needed!):**
   - Measures the **Top-3 Oracle PCK@0.10 ceiling** across all 453 validation queries (how often the true keypoint is already Peak #2 or Peak #3 in our similarity map!).
   - Draws the exact **Set-of-Mark (SoM)** visual prompt (red dot on Source + numbered circles `1, 2, 3` on Target) for real error cases where Peak #2/#3 is correct.
   - Optionally runs **zero-shot `Qwen2.5-VL-7B-Instruct` (4-bit)** live on those ambiguous queries (`RUN_VLM_DEMO = True`).

In [ ]:
# --- Settings ---
RUN = "20261123-115040_train"   # trained run ID on Drive (e.g. 0.70 dinov2-large)
BRANCH = "giuseppe"
N_EXAMPLES = 4                  # number of validation queries to trace in Part 1
TOP_K = 3                       # number of NMS candidate peaks for Part 2 (VLM SoM)
NMS_RADIUS = 10                 # minimum separation between candidate peaks (in 92x92 cells)
AMBIGUITY_GAP = 0.08            # only trigger VLM when sim(Peak1) - sim(Peak2) < AMBIGUITY_GAP
RUN_VLM_DEMO = False            # set True on Colab GPU to run zero-shot Qwen2.5-VL-7B-Instruct (4-bit)

In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ
if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/sapienza_aml_semantic_correspondence")
    if not (REPO_DIR / ".git").is_dir():
        try:
            tok = userdata.get("GITHUB_TOKEN").strip()
            url = f"https://{tok}@github.com/julpfi/sapienza_aml_semantic_correspondence.git"
        except Exception:
            url = "https://github.com/julpfi/sapienza_aml_semantic_correspondence.git"
        get_ipython().system(f"git clone -b {BRANCH} {url} {REPO_DIR}")
    get_ipython().run_line_magic("cd", str(REPO_DIR))
    get_ipython().run_line_magic("run", f"scripts/colab_setup.py --branch {BRANCH}")
else:
    REPO_DIR = Path.cwd()
    if not (REPO_DIR / "run.py").is_file() and (REPO_DIR.parent / "run.py").is_file():
        REPO_DIR = REPO_DIR.parent
    os.chdir(REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from data.datasets import build_dataset, load_pair_source_xy
from data.transforms import to_original, to_resized
from run import load_trained_weights
from src.config import get_config, resolved_device
from src.models import build_model
from src.models.matching import compute_similarity_map, locate, predict_from_features
from src.models.zoom import _stage1_pair, bbox_crop
from src.utils.run import load_run_config, run_dir_for

base_cfg = get_config()
cfg = load_run_config(RUN, base_cfg["output_dir"])
cfg["object_zoom"] = True
device = resolved_device(cfg)
run_dir = run_dir_for(cfg, RUN)
model = load_trained_weights(cfg, build_model(cfg, device), run_dir).eval()
val_ds = build_dataset(cfg, "val")
cache = model.new_cache()
print(f"Loaded {RUN} ({cfg['model_id']}) on {device} | val queries = {len(val_ds)}")

In [ ]:
# --- Pipeline tracing & Top-K NMS peak extraction helpers ---
def feature_pca_rgb(feats_chw):
    """Project [C, h, w] feature map to an [h, w, 3] RGB image via top-3 PCA components."""
    c, h, w = feats_chw.shape
    x = feats_chw.detach().cpu().float().reshape(c, -1).T  # [N, C]
    x = x - x.mean(dim=0, keepdim=True)
    _, _, v = torch.pca_lowrank(x, q=3)
    proj = (x @ v[:, :3]).reshape(h, w, 3).numpy()
    lo = np.percentile(proj, 2, axis=(0, 1), keepdims=True)
    hi = np.percentile(proj, 98, axis=(0, 1), keepdims=True)
    return np.clip((proj - lo) / np.maximum(hi - lo, 1e-6), 0.0, 1.0)

def extract_topk_peaks(sim_1hw, out_wh, offset_xy, cfg, k=3, nms_radius=10):
    """Extract top-K spatially separated local peaks from [1, H, W] similarity map.
    Returns list of dicts: {rank, score, xy_orig [2], xy_res [2], rc (r, c)}."""
    size = int(cfg["image_size"])
    s = sim_1hw[0].clone()  # [H, W]
    h, w = s.shape
    yy, xx = torch.meshgrid(torch.arange(h, device=s.device), torch.arange(w, device=s.device), indexing="ij")
    peaks = []
    for rank in range(1, k + 1):
        flat = int(s.argmax().item())
        r, c = flat // w, flat % w
        score = float(sim_1hw[0, r, c].item())
        if not np.isfinite(s[r, c].item()):
            break
        # Refine sub-pixel coordinate around (r, c) using a local mask on the original sim map
        local_mask = ((yy - r) ** 2 + (xx - c) ** 2) <= (nms_radius ** 2)
        s_local = sim_1hw[0].masked_fill(~local_mask, float("-inf"))[None]
        xy_res = locate(s_local, size, cfg["matcher"], cfg["soft_window"], cfg["soft_temperature"])[0]
        xy_orig = to_original(xy_res.cpu(), out_wh, size).numpy()[0] + np.asarray(offset_xy, dtype=np.float32)
        peaks.append({"rank": rank, "score": score, "xy_orig": xy_orig, "xy_res": xy_res.cpu().numpy(), "rc": (r, c)})
        s = s.masked_fill(local_mask, float("-inf"))
    return peaks

@torch.no_grad()
def trace_sample(model, sample, cfg, cache, top_k=3, nms_radius=10):
    """Run Stage-1 and Stage-2 explicitly and return every intermediate artifact for visualization."""
    src_wh, trg_wh = sample["src_img"].size, sample["trg_img"].size
    size = int(cfg["image_size"])
    fs1, ft1, flipped = _stage1_pair(model, sample, cfg, cache)
    ctx_xy = load_pair_source_xy(cfg, sample["pair_id"], fallback_xy=sample["src_xy"][None])
    coarse_ctx = predict_from_features(fs1, ft1, ctx_xy, src_wh, trg_wh, cfg, flipped)
    sim1 = compute_similarity_map(fs1, ft1, sample["src_xy"][None], src_wh, cfg, flipped)
    pred1_res = locate(sim1, size, cfg["matcher"], cfg["soft_window"], cfg["soft_temperature"])
    pred1 = to_original(pred1_res.cpu(), trg_wh, size).numpy()[0]

    max_area = float(cfg.get("zoom_max_area", 1.0))
    s_crop, sbox = bbox_crop(sample["src_img"], ctx_xy, cfg["zoom_margin"], cfg["zoom_min_frac"],
                             cfg["zoom_pct_low"], cfg["zoom_pct_high"], max_area=max_area)
    t_crop, tbox = bbox_crop(sample["trg_img"], coarse_ctx, cfg["zoom_margin"], cfg["zoom_min_frac"],
                             cfg["zoom_pct_low"], cfg["zoom_pct_high"], max_area=max_area)
    sc_wh = (sbox[2] - sbox[0], sbox[3] - sbox[1])
    tc_wh = (tbox[2] - tbox[0], tbox[3] - tbox[1])
    q_in = np.asarray(sample["src_xy"], dtype=np.float32) - np.array(sbox[:2], dtype=np.float32)
    inside = (0.0 <= q_in[0] < sc_wh[0]) and (0.0 <= q_in[1] < sc_wh[1])

    if inside:
        fs2 = model.image_features(s_crop, flip=flipped, cache=cache, key=("trace_s", sample["pair_id"], sbox, size))
        ft2 = model.image_features(t_crop, cache=cache, key=("trace_t", sample["pair_id"], tbox, size))
        fs2, ft2 = model.pair_features(fs2, ft2)
        sim2 = compute_similarity_map(fs2, ft2, q_in[None], sc_wh, cfg, flipped)
        peaks = extract_topk_peaks(sim2, tc_wh, tbox[:2], cfg, k=top_k, nms_radius=nms_radius)
    else:
        fs2, ft2, sim2 = fs1, ft1, sim1
        s_crop, t_crop, sbox, tbox = sample["src_img"], sample["trg_img"], (0, 0, *src_wh), (0, 0, *trg_wh)
        peaks = extract_topk_peaks(sim1, trg_wh, (0.0, 0.0), cfg, k=top_k, nms_radius=nms_radius)

    trg_gt = np.asarray(sample["trg_xys"], dtype=np.float32).reshape(-1, 2)
    L = float(sample["scale"])
    for pk in peaks:
        pk["dist_L"] = float(np.linalg.norm(trg_gt - pk["xy_orig"][None], axis=1).min() / L)
        pk["correct"] = pk["dist_L"] < 0.10

    return {
        "sample": sample, "ctx_xy": ctx_xy, "coarse_ctx": coarse_ctx,
        "fs1": fs1, "ft1": ft1, "sim1": sim1, "pred1": pred1,
        "s_crop": s_crop, "t_crop": t_crop, "sbox": sbox, "tbox": tbox,
        "fs2": fs2, "ft2": ft2, "sim2": sim2, "q_in": q_in, "peaks": peaks,
    }

## Part 1: Visual Walkthrough — The Full Path of an Image Through the Model (Without VLM)

In [ ]:
def plot_image_path(tr):
    s = tr["sample"]
    trg_gt = np.asarray(s["trg_xys"], dtype=np.float32).reshape(-1, 2)
    L = float(s["scale"])
    sbox, tbox = tr["sbox"], tr["tbox"]
    peaks = tr["peaks"]
    pred2 = peaks[0]["xy_orig"]
    d1 = float(np.linalg.norm(trg_gt - tr["pred1"][None], axis=1).min() / L)
    d2 = peaks[0]["dist_L"]

    fig, axes = plt.subplots(2, 3, figsize=(15, 8.8))
    status = "CORRECT" if peaks[0]["correct"] else ("FIXABLE AT PEAK #" + str(next((p["rank"] for p in peaks if p["correct"]), "none")))
    fig.suptitle(f"Query {s['query_id']} (pair {s['pair_id']}) | Stage-1 d/L={d1:.3f} -> Stage-2 d/L={d2:.3f} | {status}",
                 fontsize=12.5, fontweight="bold")

    # 1. Original Source + context + Stage-2 crop box
    ax = axes[0, 0]
    ax.imshow(s["src_img"])
    ax.scatter(tr["ctx_xy"][:, 0], tr["ctx_xy"][:, 1], c="cyan", s=24, edgecolors="black", lw=0.5, label="context source_xy")
    ax.scatter([s["src_xy"][0]], [s["src_xy"][1]], c="red", s=90, edgecolors="white", lw=1.5, label="query")
    ax.add_patch(Rectangle((sbox[0], sbox[1]), sbox[2]-sbox[0], sbox[3]-sbox[1], fill=False, edgecolor="yellow", lw=2, ls="--", label="Stage-2 sbox"))
    ax.set_title("1. Original Source + Zoom Box")
    ax.legend(loc="lower right", fontsize=7.5); ax.axis("off")

    # 2. DINOv2 46x46 Feature PCA (Source)
    ax = axes[0, 1]
    ax.imshow(feature_pca_rgb(tr["fs1"]))
    q_cell = to_resized(s["src_xy"][None], s["src_img"].size, tr["fs1"].shape[-1])[0]
    ax.scatter([q_cell[0]], [q_cell[1]], c="red", s=90, edgecolors="white", lw=1.5)
    ax.set_title(f"2. Stage-1 Feature Map PCA ({tr['fs1'].shape[0]}-d, {tr['fs1'].shape[1]}x{tr['fs1'].shape[2]})")
    ax.axis("off")

    # 3. Stage-1 Target + coarse context + Stage-1 sim + tbox
    ax = axes[0, 2]
    ax.imshow(s["trg_img"])
    w_t, h_t = s["trg_img"].size
    ax.imshow(tr["sim1"][0].detach().cpu().numpy(), cmap="inferno", alpha=0.45, extent=(0, w_t, h_t, 0))
    ax.scatter(tr["coarse_ctx"][:, 0], tr["coarse_ctx"][:, 1], c="cyan", s=24, edgecolors="black", lw=0.5, label="coarse matches")
    ax.add_patch(Rectangle((tbox[0], tbox[1]), tbox[2]-tbox[0], tbox[3]-tbox[1], fill=False, edgecolor="yellow", lw=2, ls="--", label="Stage-2 tbox"))
    ax.scatter([tr["pred1"][0]], [tr["pred1"][1]], c="orange", marker="^", s=85, edgecolors="black", label=f"Stage-1 pred ({d1:.2f}L)")
    ax.set_title("3. Stage-1 Coarse Sim & Target Zoom Box")
    ax.legend(loc="lower right", fontsize=7.5); ax.axis("off")

    # 4. Stage-2 Zoomed Source Crop (resized to 644x644 internally)
    ax = axes[1, 0]
    ax.imshow(tr["s_crop"])
    ax.scatter([tr["q_in"][0]], [tr["q_in"][1]], c="red", s=95, edgecolors="white", lw=1.5, label="query in crop")
    ax.set_title(f"4. Stage-2 Zoomed Source Crop ({tr['s_crop'].size[0]}x{tr['s_crop'].size[1]} -> 644)")
    ax.legend(loc="lower right", fontsize=7.5); ax.axis("off")

    # 5. Stage-2 92x92 Similarity Map + 11x11 window + Top-3 Peaks
    ax = axes[1, 1]
    sim2_np = tr["sim2"][0].detach().cpu().numpy()
    ax.imshow(sim2_np, cmap="inferno")
    r0, c0 = peaks[0]["rc"]
    half = cfg["soft_window"] // 2
    ax.add_patch(Rectangle((c0 - half, r0 - half), cfg["soft_window"], cfg["soft_window"], fill=False, edgecolor="cyan", lw=1.8, label="11x11 soft-argmax"))
    colors = ["lime", "yellow", "magenta"]
    for pk, col in zip(peaks, colors):
        r, c = pk["rc"]
        ax.scatter([c], [r], c=col, s=75, edgecolors="black", lw=1.2)
        ax.text(c + 1.5, r - 1.5, f"#{pk['rank']} ({pk['score']:.2f})", color="white", fontsize=8, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.15", fc="black", alpha=0.65, ec="none"))
    ax.set_title(f"5. Stage-2 Similarity Map ({sim2_np.shape[0]}x{sim2_np.shape[1]}) + Top-{len(peaks)} Peaks")
    ax.legend(loc="lower right", fontsize=7.5); ax.axis("off")

    # 6. Final Original Target with GT, Stage-1 vs Stage-2, and Top-3 Candidates
    ax = axes[1, 2]
    ax.imshow(s["trg_img"])
    for pt in trg_gt:
        ax.add_patch(Circle((pt[0], pt[1]), 0.10 * L, fill=False, edgecolor="lime", lw=1.6, ls="--"))
    ax.scatter(trg_gt[:, 0], trg_gt[:, 1], c="lime", s=75, edgecolors="black", label="GT accepted")
    for pk, col in zip(peaks, ["red", "yellow", "magenta"]):
        xy = pk["xy_orig"]
        ax.scatter([xy[0]], [xy[1]], c=col, marker="X" if pk["rank"]==1 else "o", s=105 if pk["rank"]==1 else 75,
                   edgecolors="black", lw=1.2, label=f"Peak #{pk['rank']} ({pk['dist_L']:.2f}L)")
    ax.set_title("6. Final Prediction Mapped to Target Pixels")
    ax.legend(loc="lower right", fontsize=7.2); ax.axis("off")

    fig.tight_layout()
    plt.show()

# Trace all 453 val queries once (reused for Part 1 & Part 2!)
all_traces = []
for i in tqdm(range(len(val_ds)), desc="Tracing val queries"):
    all_traces.append(trace_sample(model, val_ds[i], cfg, cache, top_k=TOP_K, nms_radius=NMS_RADIUS))

# Pick a balanced mix: 2 correct + 2 fixable errors (Peak #1 wrong, Peak #2 or #3 correct)
fixable = [tr for tr in all_traces if (not tr["peaks"][0]["correct"]) and any(p["correct"] for p in tr["peaks"][1:])]
correct_ex = [tr for tr in all_traces if tr["peaks"][0]["correct"]]
shown = (correct_ex[: max(1, N_EXAMPLES // 2)] + fixable[: max(1, N_EXAMPLES - N_EXAMPLES // 2)])[:N_EXAMPLES]
for tr in shown:
    plot_image_path(tr)

## Part 2A: Top-K Oracle Ceiling — How Much Room Does a Zero-Shot VLM Re-Ranker Have?

In [ ]:
n = len(all_traces)
top1 = sum(tr["peaks"][0]["correct"] for tr in all_traces)
top2 = sum(any(p["correct"] for p in tr["peaks"][:2]) for tr in all_traces)
top3 = sum(any(p["correct"] for p in tr["peaks"][:3]) for tr in all_traces)

gaps = [tr["peaks"][0]["score"] - tr["peaks"][1]["score"] if len(tr["peaks"]) > 1 else 1.0 for tr in all_traces]
ambig = [tr for tr, g in zip(all_traces, gaps) if g < AMBIGUITY_GAP]
ambig_top1 = sum(tr["peaks"][0]["correct"] for tr in ambig)
ambig_top3 = sum(any(p["correct"] for p in tr["peaks"][:3]) for tr in ambig)

print(f"All {n} validation queries:")
print(f"  Top-1 PCK@0.10 (current model): {100*top1/n:5.2f}% ({top1}/{n})")
print(f"  Top-2 Oracle PCK@0.10:          {100*top2/n:5.2f}% ({top2}/{n})  [+{100*(top2-top1)/n:.2f} pts]")
print(f"  Top-3 Oracle PCK@0.10:          {100*top3/n:5.2f}% ({top3}/{n})  [+{100*(top3-top1)/n:.2f} pts]")
print(f"\nAmbiguous subset (Peak1 - Peak2 < {AMBIGUITY_GAP}): {len(ambig)}/{n} queries ({100*len(ambig)/n:.1f}%)")
print(f"  Top-1 on ambiguous: {100*ambig_top1/max(len(ambig),1):5.1f}% -> Top-3 Oracle: {100*ambig_top3/max(len(ambig),1):5.1f}% (+{ambig_top3-ambig_top1} fixable queries!)")

## Part 2B: What a Zero-Shot Pretrained VLM Sees (Set-of-Mark Prompt) & Live Optional Demo

**Why no fine-tuning is needed:** Modern open-source VLMs like `Qwen2.5-VL-7B-Instruct` are already pretrained on millions of visual-grounding and Set-of-Mark tasks. We never ask the VLM to guess raw pixel coordinates — we only ask it to choose among the exact sub-pixel candidate peaks (`1, 2, 3`) proposed by our `0.70` DINOv2-Large model when Peak #1 and Peak #2 are close.

In [ ]:
VLM_PROMPT = (
    "Image 1 shows an object with a RED circle marking one specific semantic part. "
    "Image 2 shows another instance of the same object category (possibly rotated, flipped, or from a different 3D viewpoint) "
    "with numbered candidate points 1, 2, 3. "
    "Which candidate number (1, 2, or 3) corresponds to the EXACT same semantic part as the red circle in Image 1? "
    "Pay close attention to 3D orientation, left vs. right side, and front vs. back. "
    "Reply with ONLY the single digit (1, 2, or 3)."
)

def make_som_pair(tr):
    """Draw clean Set-of-Mark annotations on copies of source & target images for the VLM."""
    src = tr["sample"]["src_img"].copy()
    trg = tr["sample"]["trg_img"].copy()
    r_s = max(6, int(0.022 * max(src.size)))
    r_t = max(8, int(0.028 * max(trg.size)))
    ds = ImageDraw.Draw(src)
    sx, sy = tr["sample"]["src_xy"]
    ds.ellipse((sx-r_s, sy-r_s, sx+r_s, sy+r_s), fill="red", outline="white", width=3)

    dt = ImageDraw.Draw(trg)
    palette = ["#e74c3c", "#f1c40f", "#00e5ff"]
    for pk, col in zip(tr["peaks"], palette):
        tx, ty = pk["xy_orig"]
        dt.ellipse((tx-r_t, ty-r_t, tx+r_t, ty+r_t), fill=col, outline="black", width=2)
        dt.text((tx - r_t * 0.35, ty - r_t * 0.55), str(pk["rank"]), fill="black")
    return src, trg

# Show 3 real ambiguous, fixable cases (where Peak #1 was wrong and Peak #2 or #3 was correct!)
demo_cases = [tr for tr in fixable if (tr["peaks"][0]["score"] - tr["peaks"][1]["score"]) < AMBIGUITY_GAP][:3]
if not demo_cases:
    demo_cases = fixable[:3]

print("Prompt sent to zero-shot VLM:\n", VLM_PROMPT, "\n")
for tr in demo_cases:
    src_som, trg_som = make_som_pair(tr)
    true_rank = next(p["rank"] for p in tr["peaks"] if p["correct"])
    gap = tr["peaks"][0]["score"] - tr["peaks"][1]["score"]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.5, 4.3))
    ax1.imshow(src_som); ax1.set_title(f"VLM Input Image 1 (Source query {tr['sample']['query_id']})"); ax1.axis("off")
    ax2.imshow(trg_som); ax2.set_title(f"VLM Input Image 2 | Cosine picked #1 (gap={gap:.3f}), True = #{true_rank}"); ax2.axis("off")
    fig.tight_layout(); plt.show()

# Optional: actually run zero-shot Qwen2.5-VL-7B-Instruct (4-bit) on Colab GPU
if RUN_VLM_DEMO:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes", "accelerate", "qwen-vl-utils"], check=True)
    from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration
    vlm_id = "Qwen/Qwen2.5-VL-7B-Instruct"
    qcfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    vlm = Qwen2_5_VLForConditionalGeneration.from_pretrained(vlm_id, quantization_config=qcfg, device_map="auto")
    proc = AutoProcessor.from_pretrained(vlm_id)
    for tr in demo_cases:
        src_som, trg_som = make_som_pair(tr)
        msgs = [{"role": "user", "content": [
            {"type": "image", "image": src_som},
            {"type": "image", "image": trg_som},
            {"type": "text", "text": VLM_PROMPT},
        ]}]
        text = proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = proc(text=[text], images=[src_som, trg_som], return_tensors="pt").to(vlm.device)
        out = vlm.generate(**inputs, max_new_tokens=4, do_sample=False)
        ans = proc.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        true_rank = next(p["rank"] for p in tr["peaks"] if p["correct"])
        print(f"Query {tr['sample']['query_id']}: Cosine=#1 (WRONG) | Zero-shot Qwen2.5-VL={ans!r} | True=#{true_rank}")